# Data Merging — Stage 3 Cleaning 02: Weekly Deduplication

## Input
`Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_macro_daily_engineered.parquet` (34 weekly-sourced columns, forward-filled to daily by Stage 1's `merge_asof`)

## Purpose
Recovers the 34 weekly features (CFTC positioning, AAII sentiment, jobless claims, Fed H.4.1 balance sheet, Fed H.8 banking) at their native weekly cadence, undoing the daily forward-fill applied in Stage 1. These features were dropped from the Stage 2 aggregate tables in notebook 01 specifically so they could be rebuilt here.

## The Problem This Fixes
Stage 1 forward-filled each weekly source to daily via `merge_asof`, so a single weekly value repeats for ~5 trading days. Stage 3's old pipeline then z-scored that repeated daily series against a moving expanding mean/std, which manufactures a *new* z-score every day even though no new information arrived — confirmed by a boundary diagnostic showing `unique_ratio = 1.000` across all 34 features. The fix is to z-score at weekly cadence and forward-fill the *z-scores* (done in Stage 4); this notebook supplies that weekly-cadence input.

## Method
Rather than re-reading the original cleaned source files, this notebook **deduplicates** the already-merged, already-publication-lagged daily column from Panel C. Panel C already has Stage 1's publication lags correctly applied (CFTC `available_date`, claims +5d, H.4.1 +1d, H.8 +9d), so deduplication inherits that timing exactly rather than re-deriving it.

An "update" is detected as a value change (`s.ne(s.shift()) & s.notna()`), not a fixed weekly step, which correctly:
- Excludes forward-filled repeats (bit-identical to the prior value).
- Excludes the pre-June-2006 CFTC NaN block, since `NaN != NaN` is `True` under IEEE 754 and would otherwise register ~630 spurious "updates" — the explicit `& s.notna()` guards against this.
- Still catches the first genuine value once a series starts (`100.0 != NaN` is `True`).

## The One Additional Shift, and Why Only Three Features
On top of Stage 1's publication-lag shifts, this notebook applies one further **+1 trading day** shift, but only to `['fed_assets', 'tga', 'reserves']` (the H.4.1 group) — despite the variable name `H41_CLAIMS_FEATURES` and surrounding comments in the original code implying jobless claims were included too. They are not, and the code (not the comments) is correct. The reason: theme allocation groups `fed_assets`, `tga`, `reserves` (H.4.1, Thursday) together with `bank_credit`, `ci_loans` (H.8, Friday) into a single subtheme, "Fed Balance Sheet & Banking," and the sparse KAN's update mask requires every feature in a subtheme to update on the same day. The shift moves H.4.1 from Thursday to Friday to align with H.8. `initial_claims` and `continued_claims` live in a different subtheme (their own, "Weekly Unemployment Claims") with nothing to align to, so shifting them would only delay information by a day for no benefit.

## Differencing
Six features are non-stationary levels and are replaced by their week-over-week change (`_diff` suffix), computed **after** deduplication at true weekly cadence — differencing the forward-filled daily series instead would produce zero on ~4 of every 5 days:
- `tga`, `fed_assets`, `reserves` — the change captures the liquidity drain/injection (debt-ceiling artefacts, QE/QT), not the level.
- `bank_credit`, `ci_loans` — credit *growth* is the signal, not the stock of credit.
- `open_interest` — grew structurally over the sample; the flow is the signal.

## Output Shape
A **wide, sparse union**: one row per date on which *any* weekly feature updated, with each column populated only on its own update days and `NaN` elsewhere. A single dense grid isn't possible because sources update on different weekdays post-shift (Monday: CFTC; Thursday: AAII, claims; Friday: H.4.1, H.8).

**Critical consequence for Stage 4:** the union table holds only ~3 rows per week, so 52 rows is only ~17 weeks of coverage — Stage 4's per-feature warm-up must count each feature's own **observations**, not rows in this table. This notebook's summary statistics are printed specifically as the cross-check for that requirement.

## Validation / Diagnostics
- **Median gap check** — expects 7 days per source (6/8 around holidays is normal, since `merge_asof` lands releases on the next trading day).
- **Missed-release check** — flags any feature with a gap >14 days (e.g. would catch the Dec 2018–Jan 2019 government shutdown, which suspended CFTC reporting for ~5 weeks).
- **Observation-count check** — compares each feature's observation count against the expected ~52/year given its history length, flagging non-CFTC features with suspiciously few observations (possible value-change detection merging adjacent weeks).
- **Warm-up cross-check** — reports minimum observation count across all features and whether every feature clears the 52-observation threshold Stage 4 requires.
- Reports first/last valid observation and per-differenced-feature summary stats, noting the first observation of each `_diff` series is `NaN` (no prior week to difference against).

## Output
- `Data/Data_Collection/Final/Stage_3_Cleaning/weekly_raw.parquet` — sparse wide union table, one row per any-feature update date.
- `Data/Data_Collection/Final/Stage_3_Cleaning/weekly_summary.csv` — per-feature diagnostics (source, shift flag, observation count, first/last date, modal weekday, median/max gap).

In [6]:
"""
Stage 3 Cleaning — 02: Weekly Deduplication
===========================================
Recovers the 34 weekly features at their native weekly frequency.

THE PROBLEM THIS FIXES
----------------------
Stage 1 forward-filled the weekly sources to daily via merge_asof, so each
weekly value repeats for ~5 trading days. The old Stage 3 then z-scored that
daily series -- a constant numerator over a moving expanding mean and std,
producing a NEW z-score every day even though no new data arrived. The boundary
diagnostic showed unique_ratio = 1.000 for all 34, confirming it.

The fix is to z-score at weekly cadence and forward-fill the Z-SCORES, which
Stage 4 does. This notebook supplies the weekly input.

METHOD
------
Deduplicate rather than re-read the cleaned files. Panel C already has Stage 1's
publication lags correctly applied (CFTC available_date, claims +5d, H.4.1 +1d,
H.8 +9d), so deduplication inherits them exactly.

THE ONE SHIFT, AND WHY ONLY THREE FEATURES
------------------------------------------
Stage 1's publication lags convert reference date -> publication date and are
the only timing adjustment. On top of that, the old 01_prepare_datasets applied
a further +1 trading day to the H.4.1 group:

    H41_CLAIMS_FEATURES = ['fed_assets', 'tga', 'reserves']

Three features, despite the variable name and the surrounding comments claiming
claims were included too. The code is right and the comments are wrong, and the
theme allocation shows why:

    subtheme 9_7 "Fed Balance Sheet & Banking" =
        fed_assets, tga, reserves   (H.4.1, Thursday)
        bank_credit, ci_loans       (H.8,   Friday)

The shift exists purely so every feature in 9_7 updates on the same day, which
the sparse KAN's update mask requires. initial_claims and continued_claims sit
in 12_1 and are moved to their own subtheme 12_11 "Weekly Unemployment Claims",
so they have nothing to align with. Shifting them would delay information by a
day for no benefit.

OUTPUT SHAPE
------------
Wide: one row per date on which ANY weekly feature updated, each column valued
only on its own update days, NaN elsewhere. A single dense grid is impossible
because the sources update on different weekdays -- after the H.4.1 shift:

    Monday    CFTC (22)
    Thursday  AAII (5), claims (2)
    Friday    H.4.1 (3), H.8 (2)

  ** Stage 4 must count each feature's warm-up in OBSERVATIONS, not rows. **
  The union holds ~3 rows per week, so 52 rows is only ~17 weeks. Each feature
  needs 52 of ITS OWN observations. The per-feature warm-up in normalise.py does
  this; the observation counts printed below are the cross-check.

Input:   Stage_1_5_Validation_and_Feature_Engineering/panel_macro_daily_engineered.parquet
Output:  Stage_3_Cleaning/weekly_raw.parquet
"""

import numpy as np
import pandas as pd
from pathlib import Path

PANEL_C = Path('../../../Data/Data_Collection/Final'
               '/Stage_1_5_Validation_and_Feature_Engineering'
               '/panel_macro_daily_engineered.parquet')
OUT_DIR = Path('../../../Data/Data_Collection/Final/Stage_3_Cleaning')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# The 34 weekly features, grouped by source. Each group shares an update day.
GROUPS = {
    'CFTC': ['lev_long', 'lev_short', 'lev_spread', 'am_long', 'am_short',
             'am_spread', 'dealer_long', 'dealer_short', 'dealer_spread',
             'other_long', 'other_short', 'other_spread', 'open_interest',
             'lev_net', 'am_net', 'dealer_net', 'lev_net_pct', 'am_net_pct',
             'dealer_net_pct', 'lev_am_ratio', 'lev_net_chg', 'am_net_chg'],
    'AAII': ['bullish', 'neutral', 'bearish', 'bullish_8w_ma',
             'bull_bear_spread'],
    'Claims': ['initial_claims', 'continued_claims'],
    'H41': ['fed_assets', 'tga', 'reserves'],
    'H8': ['bank_credit', 'ci_loans'],
}
WEEKLY = [c for cols in GROUPS.values() for c in cols]
assert len(WEEKLY) == 34, f"Expected 34 weekly features, got {len(WEEKLY)}"

# +1 trading day, Thursday -> Friday. H.4.1 only, matching the original code
# (NOT claims -- see the docstring). Aligns subtheme 9_7 onto a single update day.
SHIFT_1D = ['fed_assets', 'tga', 'reserves']

# Non-stationary levels replaced by their one-week change. Differenced HERE, at
# weekly cadence, so the result is a genuine week-on-week change. Differencing
# the forward-filled daily series would give zero on ~4 days in 5.
#   tga            debt-ceiling artefact; the change is the liquidity drain
#   fed_assets     QE/QT; the change is the injection
#   reserves       same
#   bank_credit    credit growth, not the stock of credit
#   ci_loans       same
#   open_interest  grew structurally; the flow is the signal
DIFF = ['tga', 'fed_assets', 'reserves', 'bank_credit', 'ci_loans',
        'open_interest']


print("=" * 78)
print("STAGE 3 CLEANING — 02: WEEKLY DEDUPLICATION")
print("=" * 78)

df = pd.read_parquet(PANEL_C, columns=['date'] + WEEKLY)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

dates = df['date'].to_numpy()
n = len(df)

print(f"\n  Panel C: {n:,} daily rows, "
      f"{df['date'].min().date()} -> {df['date'].max().date()}")
print(f"  Weekly features: {len(WEEKLY)}")
print(f"  Shifted +1 trading day: {SHIFT_1D}")
print(f"  Differenced at weekly cadence: {len(DIFF)}")

# ═══════════════════════════════════════════════════════════════════════════════
# DEDUPLICATE
# ═══════════════════════════════════════════════════════════════════════════════

series, info = {}, []

for col in WEEKLY:
    s = pd.to_numeric(df[col], errors='coerce')

    # An update is a change in value. A forward-filled value is bit-identical to
    # the one before it, so exact comparison is right here.
    #
    # `& notna` is load-bearing: pandas follows IEEE 754, so NaN != NaN is TRUE.
    # Without it, CFTC's NaN block (2004-01 to 2006-06) would register ~630
    # spurious updates. With it the block is excluded, and the first real value
    # in June 2006 still registers correctly, since 100.0 != NaN is True.
    upd = s.ne(s.shift()) & s.notna()

    pos = np.flatnonzero(upd.to_numpy())
    val = s.to_numpy()[pos]

    # +1 trading day. Index arithmetic on Panel C's calendar, which IS the
    # trading calendar, so "next trading day" is just +1 and holidays are
    # handled for free. Any update on the final row is dropped.
    if col in SHIFT_1D:
        pos = pos + 1
        keep = pos < n
        pos, val = pos[keep], val[keep]

    obs = pd.Series(val, index=pd.DatetimeIndex(dates[pos]))

    name = col
    if col in DIFF:
        obs = obs.diff()          # previous element = previous week
        name = f'{col}_diff'

    series[name] = obs

    gaps = obs.index.to_series().diff().dt.days.dropna()
    info.append({
        'feature': name,
        'source': next(g for g, cols in GROUPS.items() if col in cols),
        'shifted': col in SHIFT_1D,
        'n_obs': int(obs.notna().sum()),
        'first': obs.index.min().date(),
        'last': obs.index.max().date(),
        'weekday': obs.index.day_name().value_counts().index[0],
        'median_gap': int(gaps.median()) if len(gaps) else 0,
        'max_gap': int(gaps.max()) if len(gaps) else 0,
    })

# Union of every update date, sorted
weekly = pd.DataFrame(index=sorted({d for s in series.values() for d in s.index}))
for name, s in series.items():
    weekly[name] = s
weekly.index.name = 'date'
weekly = weekly.reset_index()

info = pd.DataFrame(info)

# ═══════════════════════════════════════════════════════════════════════════════
# REPORT
# ═══════════════════════════════════════════════════════════════════════════════

print(f"\n  Result: {len(weekly):,} rows x {weekly.shape[1] - 1} features")
print(f"  Dates:  {weekly['date'].min().date()} -> {weekly['date'].max().date()}")
print(f"  Compression: {n:,} daily rows -> {len(weekly):,} union rows")

print(f"\n  BY SOURCE")
print(f"    {'source':<8} {'feats':>5} {'obs':>6} {'weekday':<10} {'shift':>6} "
      f"{'first':>12} {'gap':>4}")
print(f"    {'-' * 58}")
for src in GROUPS:
    g = info[info['source'] == src]
    print(f"    {src:<8} {len(g):>5} {int(g['n_obs'].median()):>6} "
          f"{g['weekday'].iloc[0]:<10} {'+1d' if g['shifted'].iloc[0] else '':>6} "
          f"{str(g['first'].min()):>12} {int(g['median_gap'].median()):>4}")

print(f"\n  Median gap should be 7 days for every source. Values of 6 or 8")
print(f"  around a holiday are normal -- merge_asof lands the release on the")
print(f"  next trading day.")

# ── Gap check: a missed release ─────────────────────────────────────────────
# 7 days normal, up to ~10 across a holiday. Beyond 14 means a week was skipped.
# The Dec 2018 - Jan 2019 shutdown suspended CFTC reporting for ~5 weeks, which
# would appear here.
big = info[info['max_gap'] > 14]
if len(big):
    print(f"\n  ** {len(big)} features with a gap >14 days -- a release was "
          f"missed **")
    for _, r in big.sort_values('max_gap', ascending=False).iterrows():
        print(f"    {r['feature']:<24s} max gap {r['max_gap']:>4d} days")
    print(f"  Check whether this is a genuine reporting suspension.")
else:
    print(f"  No feature has a gap beyond 14 days")

# ── Observation counts ──────────────────────────────────────────────────────
n_years = (weekly['date'].max() - weekly['date'].min()).days / 365.25
print(f"\n  OBSERVATION COUNTS")
print(f"    Expected ~{n_years * 52:.0f} for a full-history weekly series "
      f"({n_years:.1f} years)")
print(f"    CFTC starts June 2006, so ~{(2024.9 - 2006.5) * 52:.0f} is right there")

# A weekly series reporting the identical value two weeks running would be
# missed by value-change detection, showing as fewer observations than weeks.
low = info[(info['n_obs'] < 0.85 * n_years * 52) & (info['source'] != 'CFTC')]
if len(low):
    print(f"\n    ** {len(low)} non-CFTC features with unexpectedly few "
          f"observations -- value-change detection may be merging weeks **")
    for _, r in low.iterrows():
        print(f"      {r['feature']:<24s} {r['n_obs']:>5d} obs from {r['first']}")
else:
    print(f"    No non-CFTC feature has suspiciously few observations")

# ── Warm-up cross-check for Stage 4 ─────────────────────────────────────────
print(f"\n  WARM-UP CROSS-CHECK")
print(f"    Stage 4 needs 52 OBSERVATIONS per feature, not 52 rows. The union")
print(f"    holds ~{len(weekly) / n_years:.0f} rows/year against ~52")
print(f"    observations/feature/year.")
print(f"    min obs across features: {info['n_obs'].min()} "
      f"({info.loc[info['n_obs'].idxmin(), 'feature']})")
print(f"    all features clear 52:   {bool((info['n_obs'] >= 52).all())}")

# ── Differenced features ────────────────────────────────────────────────────
print(f"\n  DIFFERENCED FEATURES")
for c in DIFF:
    s = weekly[f'{c}_diff']
    print(f"    {c + '_diff':<20s} {int(s.notna().sum()):>5d} obs   "
          f"mean {s.mean():>13.2f}   sd {s.std():>13.2f}")
print(f"    The first observation of each is NaN -- no prior week to difference.")

# ═══════════════════════════════════════════════════════════════════════════════
# SAVE
# ═══════════════════════════════════════════════════════════════════════════════

weekly.to_parquet(OUT_DIR / 'weekly_raw.parquet', index=False, engine='pyarrow')
info.to_csv(OUT_DIR / 'weekly_summary.csv', index=False)

print(f"\n  Saved: weekly_raw.parquet  ({len(weekly):,} x {weekly.shape[1]})")
print(f"  Saved: weekly_summary.csv")
print(f"\n  Next: 03_transforms")

STAGE 3 CLEANING — 02: WEEKLY DEDUPLICATION

  Panel C: 5,285 daily rows, 2004-01-02 -> 2024-12-31
  Weekly features: 34
  Shifted +1 trading day: ['fed_assets', 'tga', 'reserves']
  Differenced at weekly cadence: 6

  Result: 3,107 rows x 34 features
  Dates:  2004-01-02 -> 2024-12-30
  Compression: 5,285 daily rows -> 3,107 union rows

  BY SOURCE
    source   feats    obs weekday     shift        first  gap
    ----------------------------------------------------------
    CFTC        22    968 Monday              2006-06-19    7
    AAII         5   1094 Thursday            2004-01-02    7
    Claims       2   1072 Thursday            2004-01-08    7
    H41          3   1094 Friday        +1d   2004-01-09    7
    H8           2   1093 Friday              2004-01-16    7

  Median gap should be 7 days for every source. Values of 6 or 8
  around a holiday are normal -- merge_asof lands the release on the
  next trading day.

  ** 2 features with a gap >14 days -- a release was miss